# AcuDock Pro

**Full-featured molecular docking with optional CNN rescoring.**

AcuDock Pro combines **AutoDock Vina** with **Gnina CNN rescoring** for more accurate
binding pose predictions. It also supports **Uni-Dock GPU acceleration** for faster screening.

### Features

- **Three scoring modes:** Vina only, Vina + Gnina CNN, or Consensus (z-score weighted)
- **GPU acceleration:** Optional Uni-Dock mode for 1000x+ speedup on batch docking
- **3D visualization** of binding poses with interactive viewer
- **Batch screening** to test multiple molecules at once
- **Multi-protein docking** to test one molecule against several targets in parallel

### Getting Started

1. Click **Runtime > Run all** to run all cells
2. The first cell installs dependencies (~2-3 min) and the runtime will **automatically restart**
3. After restart, click **Runtime > Run all** again
4. The interactive panel will appear below

### What is CNN Rescoring?

Vina uses a math formula to estimate binding energy. Gnina adds a neural network
(trained on thousands of known protein-drug pairs) that re-evaluates each pose.
This improves the success rate of finding the correct binding pose from ~58% to ~73%.
To use Gnina, switch your Colab runtime to GPU (**Runtime > Change runtime type > T4 GPU**).

---

*License: MIT | Platform: Google Colab | Engines: AutoDock Vina, Gnina, Uni-Dock*


In [ ]:
#@title Step 1: Install Dependencies (run once, then runtime restarts)
# === Step 1: Install Dependencies ===
# After completion, the runtime restarts. Skip this cell afterward.

!pip install -q vina meeko gemmi rdkit prody py3Dmol openbabel-wheel pdbfixer pandas numpy scipy ipywidgets matplotlib reportlab prolif

# Download Gnina binary for CNN rescoring (~1.4 GB, takes ~1-2 min)
!wget https://github.com/gnina/gnina/releases/download/v1.3.2/gnina.1.3.2 -O /content/gnina && chmod +x /content/gnina && echo "Gnina installed (v1.3.2)" || echo "Gnina download failed — check network"

# Clone AcuDock repo for shared utilities
!git clone https://github.com/Grimlock5310/AcuDock.git /content/AcuDock 2>/dev/null || (cd /content/AcuDock && git pull)

# Install Uni-Dock for GPU-accelerated docking (falls back to Vina if no GPU)
# Uncomment the next line for 1000x+ speedup on NVIDIA GPUs (compute capability >= 7.0):
!wget -q https://github.com/dptech-corp/Uni-Dock/releases/download/1.1.0/unidock-1.1.0-cuda120-linux-x86_64 -O /usr/local/bin/unidock && chmod +x /usr/local/bin/unidock && echo "Uni-Dock GPU v1.1.0 installed" || echo "Uni-Dock download failed"

import os
os.kill(os.getpid(), 9)

## Interactive Docking Interface

The cells below load the docking tools and display the interactive panel.
Use the **tabs** to switch between single docking, batch screening, and multi-protein docking.
Adjust sliders and dropdowns to configure your run, then click the action button.


In [ ]:
#@title Step 2: Load Libraries
# === Step 2: Imports ===
import warnings
warnings.filterwarnings('ignore')

import os, sys, io
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Draw
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

sys.path.insert(0, '/content/AcuDock')
import acudock_utils as utils

WORK_DIR = '/content/acudock_pro'
os.makedirs(WORK_DIR, exist_ok=True)

# Check engines
print('AcuDock Pro loaded.')
print(utils.get_docking_engine_status())
gnina_ok = os.path.isfile('/content/gnina') and os.access('/content/gnina', os.X_OK)
print(f'Gnina CNN: {"Available" if gnina_ok else "Not installed"}')
try:
    import acudock_report as report
except ImportError:
    report = None
    print('Note: reportlab not installed. PDF reports unavailable.')


In [ ]:
#@title Step 3: Launch Interactive Interface
# === Step 3: Launch Interactive Interface ===
import ipywidgets as widgets
from IPython.display import display, HTML, FileLink, clear_output
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt


def run_pro_docking(pdb_id, smiles, lig_name, scoring_mode, engine,
                     exhaustiveness, n_poses, box_size, residues_str,
                     alpha, output_area):
    """Full AcuDock Pro docking pipeline with optional CNN rescoring."""
    with output_area:
        clear_output(wait=True)
        print('Starting AcuDock Pro docking...')

    try:
        pdb_id = pdb_id.strip().upper()
        smiles = smiles.strip()
        lig_name = lig_name.strip() or 'ligand'

        if not pdb_id:
            with output_area:
                print('Error: Enter a PDB ID.')
            return
        if not smiles or Chem.MolFromSmiles(smiles) is None:
            with output_area:
                print('Error: Invalid SMILES.')
            return

        with output_area:
            print(f'[1/6] Preparing protein {pdb_id}...')
        protein_pdb = utils.prepare_protein(pdb_id, output_dir=WORK_DIR)
        receptor_pdbqt = utils.pdb_to_pdbqt(protein_pdb)

        with output_area:
            print(f'[2/6] Preparing ligand {lig_name}...')
        ligand_pdbqt, lig_mol = utils.prepare_ligand(smiles, name=lig_name, output_dir=WORK_DIR)
        props = utils.get_ligand_properties(smiles)

        violations = sum([props['MW'] > 500, props['LogP'] > 5,
                          props['HBD'] > 5, props['HBA'] > 10])
        with output_area:
            print(f'       MW={props["MW"]} LogP={props["LogP"]} HBD={props["HBD"]} HBA={props["HBA"]} QED={props.get("QED","?")} TPSA={props.get("TPSA","?")}')
            print(f'       Lipinski: {"PASS" if violations <= 1 else "FAIL"} ({violations}/4 violations)')

        with output_area:
            print('[3/6] Defining search box...')
        residues = None
        if residues_str.strip():
            residues = [int(r.strip()) for r in residues_str.split(',') if r.strip().isdigit()]
        if residues:
            center = utils.get_binding_site_center(protein_pdb, chain='A', residues=residues)
        else:
            try:
                site_info = utils.detect_binding_site(pdb_id, output_dir=WORK_DIR)
                center = site_info['center']
                with output_area:
                    print(f'       Auto-detected binding site near {site_info.get("het_name","?")} ({site_info["method"]})')
            except Exception:
                center = utils.get_binding_site_center(protein_pdb, chain='A', residues=None)
        box = [int(box_size)] * 3
        with output_area:
            print(f'       Center: [{center[0]:.1f}, {center[1]:.1f}, {center[2]:.1f}]')

        with output_area:
            print(f'[4/6] Running Vina docking (exhaustiveness={int(exhaustiveness)})...')
            print('       This may take 1-5 minutes...')
        _, energies_arr, poses_path = utils.run_vina(
            receptor_pdbqt, ligand_pdbqt,
            center=center, box_size=box,
            exhaustiveness=int(exhaustiveness), n_poses=int(n_poses)
        )
        energies_raw = [(e[0], e[1], e[2]) for e in energies_arr]
        with output_area:
            print(f'       {len(energies_raw)} poses | Best Vina: {energies_raw[0][0]:.2f} kcal/mol')

        gnina_scores = None
        if scoring_mode in ['Vina + Gnina CNN', 'Consensus']:
            with output_area:
                print('[5/6] Running Gnina CNN rescoring...')
            gnina_raw = utils.run_gnina_rescore(receptor_pdbqt, poses_path, output_dir=WORK_DIR)
            if gnina_raw:
                gnina_scores = [s['value'] for s in gnina_raw if s['metric'] == 'CNNaffinity']
                if not gnina_scores:
                    gnina_scores = None
                    with output_area:
                        print('       No CNN affinity scores parsed.')
                else:
                    with output_area:
                        print(f'       {len(gnina_scores)} CNN scores obtained.')
            else:
                with output_area:
                    print('       Gnina unavailable, using Vina scores only.')
        else:
            with output_area:
                print('[5/6] Skipping CNN rescoring (Vina-only mode).')

        with output_area:
            print('[6/6] Building results and visualizations...')

        R, T = 1.987e-3, 298.15

        if scoring_mode == 'Consensus' and gnina_scores and len(gnina_scores) == len(energies_raw):
            results_df = utils.consensus_score(energies_raw, gnina_scores, alpha=alpha)
        else:
            results_df = pd.DataFrame({
                'Pose': range(1, len(energies_raw) + 1),
                'Vina Score': [e[0] for e in energies_raw],
                'RMSD_lb': [round(e[1], 2) for e in energies_raw],
                'RMSD_ub': [round(e[2], 2) for e in energies_raw],
                'Est. Kd (uM)': [round(np.exp(e[0] / (R * T)) * 1e6, 4) for e in energies_raw],
            })
            if gnina_scores and len(gnina_scores) == len(energies_raw):
                results_df['CNN Affinity'] = gnina_scores

        # 2D image
        mol_2d = Chem.MolFromSmiles(smiles)
        img = Draw.MolToImage(mol_2d, size=(300, 200))
        img_buf = io.BytesIO()
        img.save(img_buf, format='PNG')
        img_buf.seek(0)
        img_path = os.path.join(WORK_DIR, f'{lig_name}_2d.png')
        with open(img_path, 'wb') as f:
            f.write(img_buf.read())

        # 3D viewers will be created at display time

        csv_path = os.path.join(WORK_DIR, f'{lig_name}_{pdb_id}_pro_results.csv')
        results_df.to_csv(csv_path, index=False)

        top_score = energies_raw[0][0]
        props = utils.get_ligand_properties(smiles, docking_score=top_score)

        # ProLIF interactions
        interactions_html = None
        interactions_df = None
        try:
            ifp = utils.compute_interaction_fingerprint(protein_pdb, poses_path, pose_index=0, smiles=smiles)
            if ifp is not None:
                interactions_df = ifp.get('summary_df')
                interactions_html = utils.interaction_fingerprint_to_html(interactions_df)
        except Exception:
            pass

        with output_area:
            clear_output(wait=True)
            print(f'=== DOCKING COMPLETE ===')
            print(f'Protein: {pdb_id} | Ligand: {lig_name} | Scoring: {scoring_mode}')
            if scoring_mode == 'Consensus':
                print(f'Consensus alpha: {alpha:.2f} (Vina={alpha:.0%}, Gnina={1-alpha:.0%})')
            print(f'Best: {top_score:.2f} kcal/mol | {utils.score_interpretation(top_score)}')
            print()
            print(f'  MW={props["MW"]}  LogP={props["LogP"]}  HBD={props["HBD"]}  HBA={props["HBA"]}  RotBonds={props.get("RotBonds","?")}')
            print(f'  TPSA={props.get("TPSA","?")}  QED={props.get("QED","?")}  SA={props.get("SA_Score","?")}  PAINS={props.get("PAINS_Count",0)}')
            if props.get('LE') is not None:
                print(f'  LE={props["LE"]}  LLE={props.get("LLE","?")}')
            print(f'  Lipinski: {"PASS" if violations <= 1 else "FAIL"} ({violations}/4 violations)')
            print()
            from IPython.display import Image as IPyImage
            display(IPyImage(filename=img_path))
            print()
            display(HTML('<h4>Docking Results</h4>'))
            display(results_df)
            print()
            display(FileLink(csv_path, result_html_prefix='Download CSV: '))

            if interactions_html:
                print()
                display(HTML('<h4>Protein-Ligand Interactions (ProLIF)</h4>'))
                display(HTML(interactions_html))

            print()
            display(HTML('<h4>3D Viewer — Top Pose</h4>'))
            _v_out1 = widgets.Output()
            display(_v_out1)
            print()
            display(HTML('<h4>Multi-Pose Overlay (Top 3)</h4>'))
            _v_out2 = widgets.Output()
            display(_v_out2)
            with _v_out1:
                view = utils.visualize_pose(protein_pdb, poses_path, pose_index=0)
                view.show()
            with _v_out2:
                multi_view = utils.visualize_multi_poses(protein_pdb, poses_path, n_poses=3)
                multi_view.show()

            # PDF Report button
            if report is not None:
                print()
                _pdf_btn = widgets.Button(description='Download PDF Report',
                                          button_style='info', icon='file-pdf-o',
                                          layout=widgets.Layout(width='220px'))
                _pdf_output = widgets.Output()
                def _on_pdf(btn, _pdb=pdb_id, _name=lig_name, _smi=smiles,
                            _rdf=results_df, _props=props, _ppdb=protein_pdb,
                            _poses=poses_path, _idf=interactions_df):
                    btn.disabled = True
                    btn.description = 'Generating...'
                    try:
                        view_imgs = utils.capture_3d_views(_ppdb, _poses, WORK_DIR)
                        pdf_path = os.path.join(WORK_DIR, f'{_name}_{_pdb}_pro_report.pdf')
                        report.generate_single_dock_pdf(
                            pdf_path, _pdb, _name, _smi, _rdf, _props,
                            image_paths=view_imgs, interactions_df=_idf)
                        with _pdf_output:
                            clear_output(wait=True)
                            display(FileLink(pdf_path, result_html_prefix='PDF Ready: '))
                    except Exception as ex:
                        with _pdf_output:
                            print(f'PDF error: {ex}')
                    finally:
                        btn.disabled = False
                        btn.description = 'Download PDF Report'
                _pdf_btn.on_click(_on_pdf)
                display(_pdf_btn)
                display(_pdf_output)

    except Exception as e:
        with output_area:
            print(f'\nERROR: {str(e)}')
            import traceback
            traceback.print_exc()


def run_pro_batch(pdb_id, compounds_text, scoring_mode, engine,
                   exhaustiveness, box_size, residues_str, alpha, output_area):
    """Batch screening with optional CNN rescoring."""
    with output_area:
        clear_output(wait=True)
        print('Starting batch screening...')

    try:
        pdb_id = pdb_id.strip().upper()
        if not pdb_id:
            with output_area:
                print('Error: Enter a PDB ID.')
            return

        compounds = []
        for line in compounds_text.strip().split('\n'):
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split(',', 1) if ',' in line else line.split('\t', 1)
            if len(parts) == 2:
                name, smi = parts[0].strip(), parts[1].strip()
            else:
                smi = parts[0].strip()
                name = f'Compound_{len(compounds)+1}'
            if Chem.MolFromSmiles(smi) is not None:
                compounds.append((name, smi))

        if not compounds:
            with output_area:
                print('Error: No valid compounds.')
            return

        with output_area:
            print(f'{len(compounds)} valid compounds.')
            print(f'Preparing protein {pdb_id}...')
        protein_pdb = utils.prepare_protein(pdb_id, output_dir=WORK_DIR)
        receptor_pdbqt = utils.pdb_to_pdbqt(protein_pdb)

        residues = None
        if residues_str.strip():
            residues = [int(r.strip()) for r in residues_str.split(',') if r.strip().isdigit()]
        if residues:
            center = utils.get_binding_site_center(protein_pdb, chain='A', residues=residues)
        else:
            try:
                site_info = utils.detect_binding_site(pdb_id, output_dir=WORK_DIR)
                center = site_info['center']
                with output_area:
                    print(f'  Auto-detected binding site near {site_info.get("het_name","?")}')
            except Exception:
                center = utils.get_binding_site_center(protein_pdb, chain='A', residues=None)
        box = [int(box_size)] * 3
        batch_exh = int(exhaustiveness)
        use_unidock = 'Uni-Dock' in engine and utils.check_unidock_available()

        with output_area:
            print('Preparing ligands...')
        prepared = []
        for i, (name, smi) in enumerate(compounds):
            try:
                lig_pdbqt, _ = utils.prepare_ligand(smi, name=name, output_dir=WORK_DIR)
                prepared.append((name, smi, lig_pdbqt))
            except Exception:
                prepared.append((name, smi, None))
        with output_area:
            print(f'{len([p for p in prepared if p[2]])} ligands prepared.')

        batch_scores = {}
        valid_pdbqts = [p[2] for p in prepared if p[2]]

        if use_unidock and len(valid_pdbqts) >= 2:
            with output_area:
                print(f'Running Uni-Dock GPU batch ({len(valid_pdbqts)} ligands)...')
            try:
                ud_results = utils.run_unidock(
                    receptor_pdbqt, valid_pdbqts,
                    center=center, box_size=box,
                    exhaustiveness=batch_exh, num_modes=5,
                    output_dir=WORK_DIR
                )
                for basename, scores in ud_results.items():
                    if scores and scores[0][0] < 0:
                        batch_scores[basename] = scores[0][0]
                with output_area:
                    print(f'Uni-Dock: {len(batch_scores)} valid results.')
            except Exception as e:
                with output_area:
                    print(f'Uni-Dock failed: {e}. Falling back to Vina.')

        remaining = [(n, s, p) for n, s, p in prepared
                     if p and os.path.basename(p) not in batch_scores]
        for i, (name, smi, lig_pdbqt) in enumerate(remaining):
            with output_area:
                print(f'Vina {i+1}/{len(remaining)}: {name}...')
            try:
                _, en, _ = utils.run_vina(
                    receptor_pdbqt, lig_pdbqt,
                    center=center, box_size=box,
                    exhaustiveness=batch_exh, n_poses=5
                )
                batch_scores[os.path.basename(lig_pdbqt)] = en[0][0] if len(en) > 0 else None
            except Exception:
                batch_scores[os.path.basename(lig_pdbqt)] = None

        R, T = 1.987e-3, 298.15
        results = []
        for name, smi, lig_pdbqt in prepared:
            best = batch_scores.get(os.path.basename(lig_pdbqt) if lig_pdbqt else '', None)
            mp = utils.get_ligand_properties(smi, docking_score=best)
            results.append({
                'Name': name, 'SMILES': smi,
                'Score': round(best, 2) if best else None,
                'Est. Kd (uM)': round(np.exp(best / (R * T)) * 1e6, 4) if best else None,
                'MW': mp.get('MW'), 'LogP': mp.get('LogP'),
                'QED': mp.get('QED'), 'SA': mp.get('SA_Score'),
                'PAINS': mp.get('PAINS_Count', 0), 'LE': mp.get('LE'),
                'Interpretation': utils.score_interpretation(best) if best else 'Failed',
            })

        batch_df = pd.DataFrame(results).sort_values('Score', ascending=True).reset_index(drop=True)
        batch_df.insert(0, 'Rank', range(1, len(batch_df) + 1))

        valid = batch_df.dropna(subset=['Score'])
        fig, ax = plt.subplots(figsize=(10, max(3, len(valid) * 0.4)))
        colors = ['#2ecc71' if s <= -7 else '#f39c12' if s <= -5 else '#e74c3c'
                  for s in valid['Score']]
        ax.barh(valid['Name'], valid['Score'], color=colors, edgecolor='black', linewidth=0.5)
        ax.set_xlabel('Docking Score (kcal/mol)')
        ax.set_title(f'Batch Screening: {pdb_id}')
        ax.axvline(x=-7, color='gray', linestyle='--', alpha=0.5)
        ax.invert_yaxis()
        plt.tight_layout()
        chart_path = os.path.join(WORK_DIR, 'batch_chart.png')
        fig.savefig(chart_path, dpi=120, bbox_inches='tight')
        plt.close(fig)

        csv_path = os.path.join(WORK_DIR, f'batch_{pdb_id}_pro.csv')
        batch_df.to_csv(csv_path, index=False)

        with output_area:
            clear_output(wait=True)
            print(f'=== BATCH COMPLETE ===')
            print(f'Protein: {pdb_id} | {len(compounds)} compounds | Scoring: {scoring_mode}')
            if scoring_mode == 'Consensus':
                print(f'Consensus alpha: {alpha:.2f} (Vina={alpha:.0%}, Gnina={1-alpha:.0%})')
            print(f'Best: {batch_df["Score"].min():.2f} kcal/mol')
            print()
            display(batch_df)
            print()
            from IPython.display import Image as IPyImage
            display(IPyImage(filename=chart_path))
            print()
            display(FileLink(csv_path, result_html_prefix='Download CSV: '))

            if report is not None:
                print()
                _bpdf_btn = widgets.Button(description='Download PDF Report',
                                           button_style='info', icon='file-pdf-o',
                                           layout=widgets.Layout(width='220px'))
                _bpdf_out = widgets.Output()
                def _on_bpdf(btn, _pdb=pdb_id, _bdf=batch_df, _cp=chart_path):
                    btn.disabled = True
                    btn.description = 'Generating...'
                    try:
                        pdf_path = os.path.join(WORK_DIR, f'batch_{_pdb}_pro_report.pdf')
                        report.generate_batch_pdf(pdf_path, _pdb, _bdf, chart_path=_cp)
                        with _bpdf_out:
                            clear_output(wait=True)
                            display(FileLink(pdf_path, result_html_prefix='PDF Ready: '))
                    except Exception as ex:
                        with _bpdf_out:
                            print(f'PDF error: {ex}')
                    finally:
                        btn.disabled = False
                        btn.description = 'Download PDF Report'
                _bpdf_btn.on_click(_on_bpdf)
                display(_bpdf_btn)
                display(_bpdf_out)

    except Exception as e:
        with output_area:
            print(f'\nERROR: {str(e)}')
            import traceback
            traceback.print_exc()


def run_pro_multi_protein(smiles, lig_name, pdb_text, box_size, exhaustiveness, engine,
                           scoring_mode, alpha, output_area):
    """Multi-protein docking with optional CNN rescoring."""
    with output_area:
        clear_output(wait=True)
        print('Starting multi-protein docking...')

    try:
        smiles = smiles.strip()
        lig_name = lig_name.strip() or 'ligand'

        if not smiles or Chem.MolFromSmiles(smiles) is None:
            with output_area:
                print('Error: Invalid SMILES string.')
            return

        pdb_ids = []
        residues_map = {}
        for line in pdb_text.strip().split('\n'):
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            if ':' in line:
                pdb_part, res_part = line.split(':', 1)
                pdb_id = pdb_part.strip().upper()
                res_list = [int(r.strip()) for r in res_part.split(',') if r.strip().isdigit()]
                if res_list:
                    residues_map[pdb_id] = res_list
            else:
                pdb_id = line.strip().upper()
            if pdb_id:
                pdb_ids.append(pdb_id)

        if not pdb_ids:
            with output_area:
                print('Error: No PDB IDs found.')
            return

        with output_area:
            print(f'Docking {lig_name} against {len(pdb_ids)} proteins...')

        def progress_cb(current, total, pdb_id, status):
            with output_area:
                print(f'  [{current+1}/{total}] {pdb_id}: {status}')

        use_unidock = 'Uni-Dock' in engine and utils.check_unidock_available()
        results_df, best_pdb, best_pdb_path, best_poses = utils.dock_multi_protein(
            smiles, pdb_ids, name=lig_name,
            residues_map=residues_map,
            box_size=int(box_size),
            exhaustiveness=int(exhaustiveness),
            output_dir=WORK_DIR,
            progress_callback=progress_cb,
            use_unidock=use_unidock
        )

        csv_path = os.path.join(WORK_DIR, f'{lig_name}_multi_protein_pro.csv')
        results_df.to_csv(csv_path, index=False)

        valid = results_df.dropna(subset=['Best_Score'])
        fig, ax = plt.subplots(figsize=(10, max(3, len(valid) * 0.5)))
        colors = ['#2ecc71' if s <= -7 else '#f39c12' if s <= -5 else '#e74c3c'
                  for s in valid['Best_Score']]
        ax.barh(valid['PDB_ID'], valid['Best_Score'], color=colors,
                edgecolor='black', linewidth=0.5)
        ax.set_xlabel('Docking Score (kcal/mol)')
        ax.set_title(f'{lig_name} vs Multiple Proteins')
        ax.axvline(x=-7, color='gray', linestyle='--', alpha=0.5, label='Moderate threshold')
        ax.legend()
        ax.invert_yaxis()
        plt.tight_layout()
        chart_path = os.path.join(WORK_DIR, 'multi_protein_chart.png')
        fig.savefig(chart_path, dpi=120, bbox_inches='tight')
        plt.close(fig)

        has_viewer = best_pdb and best_pdb_path and best_poses

        with output_area:
            clear_output(wait=True)
            print(f'=== MULTI-PROTEIN DOCKING COMPLETE ===')
            print(f'{lig_name} docked against {len(pdb_ids)} proteins | Scoring: {scoring_mode}')
            if best_pdb:
                best_row = results_df[results_df['PDB_ID'] == best_pdb].iloc[0]
                print(f'Best target: {best_pdb} ({best_row["Best_Score"]:.2f} kcal/mol)')
            print()
            display(results_df[['PDB_ID', 'Best_Score', 'Est_Kd_uM', 'Interpretation',
                                'Num_Poses', 'Prep_Time_s', 'Dock_Time_s', 'Engine', 'Error']])
            print()
            from IPython.display import Image as IPyImage
            display(IPyImage(filename=chart_path))
            display(FileLink(csv_path, result_html_prefix='Download CSV: '))
            if has_viewer:
                print()
                display(HTML(f'<h4>Best Target: {best_pdb}</h4>'))
                _v_out = widgets.Output()
                display(_v_out)
                with _v_out:
                    view = utils.visualize_pose(best_pdb_path, best_poses, pose_index=0)
                    view.show()

            if report is not None:
                print()
                _mpdf_btn = widgets.Button(description='Download PDF Report',
                                           button_style='info', icon='file-pdf-o',
                                           layout=widgets.Layout(width='220px'))
                _mpdf_out = widgets.Output()
                def _on_mpdf(btn, _name=lig_name, _smi=smiles, _rdf=results_df, _cp=chart_path):
                    btn.disabled = True
                    btn.description = 'Generating...'
                    try:
                        pdf_path = os.path.join(WORK_DIR, f'{_name}_multi_pro_report.pdf')
                        report.generate_multi_protein_pdf(pdf_path, _name, _smi, _rdf, chart_path=_cp)
                        with _mpdf_out:
                            clear_output(wait=True)
                            display(FileLink(pdf_path, result_html_prefix='PDF Ready: '))
                    except Exception as ex:
                        with _mpdf_out:
                            print(f'PDF error: {ex}')
                    finally:
                        btn.disabled = False
                        btn.description = 'Download PDF Report'
                _mpdf_btn.on_click(_on_mpdf)
                display(_mpdf_btn)
                display(_mpdf_out)

    except Exception as e:
        with output_area:
            print(f'\nERROR: {str(e)}')
            import traceback
            traceback.print_exc()


# ---------------------------------------------------------------------------
# Build ipywidgets Interface
# ---------------------------------------------------------------------------

DEFAULT_COMPOUNDS = """Aspirin, CC(=O)Oc1ccccc1C(=O)O
Ibuprofen, CC(C)Cc1ccc(cc1)C(C)C(=O)O
Caffeine, Cn1c(=O)c2c(ncn2C)n(C)c1=O
Acetaminophen, CC(=O)Nc1ccc(O)cc1
Naproxen, COc1ccc2cc(ccc2c1)C(C)C(=O)O
Metformin, CN(C)C(=N)NC(=N)N
Celecoxib, Cc1ccc(-c2cc(C(F)(F)F)nn2-c2ccc(S(N)(=O)=O)cc2)cc1
Diclofenac, OC(=O)Cc1ccccc1Nc1c(Cl)cccc1Cl
Atorvastatin, CC(C)c1n(CC[C@@H](O)C[C@@H](O)CC(=O)O)c(c2ccc(F)cc2)c(c1c1ccccc1)C(=O)Nc1ccccc1
Omeprazole, COc1ccc2[nH]c(nc2c1)S(=O)Cc1ncc(C)c(OC)c1C"""

style = {'description_width': '180px'}
layout_input = widgets.Layout(width='95%')

# === Single Docking Tab ===
sd_pdb = widgets.Text(value='1HSG', description='PDB ID:', style=style, layout=layout_input)
sd_residues = widgets.Text(value='23,24,25,26,27,28,29,30',
                            description='Active Site Residues:', style=style, layout=layout_input)
sd_smiles = widgets.Textarea(
    value='CC(C)(C)NC(=O)[C@@H]1C[C@@H]2CCCN2C(=O)[C@H](CC2=CC=CC=C2)NC(=O)[C@@H](CC2=CC=C(O)C=C2)N1',
    description='Ligand SMILES:', style=style, layout=widgets.Layout(width='95%', height='60px')
)
sd_name = widgets.Text(value='Indinavir', description='Ligand Name:', style=style, layout=layout_input)
sd_scoring = widgets.RadioButtons(options=['Vina', 'Vina + Gnina CNN', 'Consensus'],
                                   value='Vina', description='Scoring Mode:', style=style)
sd_alpha = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05,
                                description='Consensus Alpha:', style=style, layout=layout_input,
                                readout_format='.2f')
sd_alpha_label = widgets.HTML(
    '<small>Alpha controls Vina vs Gnina weight: 0.0 = Gnina only, 0.5 = equal, 1.0 = Vina only</small>')
sd_alpha_box = widgets.VBox([sd_alpha, sd_alpha_label],
                             layout=widgets.Layout(width='95%'))
sd_alpha_box.layout.display = 'none'  # hidden until Consensus selected

def _toggle_sd_alpha(change):
    sd_alpha_box.layout.display = '' if change['new'] == 'Consensus' else 'none'
sd_scoring.observe(_toggle_sd_alpha, names='value')

sd_engine = widgets.RadioButtons(options=['Vina (CPU)'], value='Vina (CPU)',
                             description='Engine:', style=style, layout=layout_input)
sd_exhaust = widgets.IntSlider(value=8, min=8, max=128, step=8,
                                description='Exhaustiveness:', style=style, layout=layout_input)
sd_poses = widgets.IntSlider(value=20, min=5, max=50, step=5,
                              description='Max Poses:', style=style, layout=layout_input)
sd_box = widgets.IntSlider(value=20, min=15, max=40, step=5,
                            description='Box Size (A):', style=style, layout=layout_input)
sd_btn = widgets.Button(description='Run Docking', button_style='primary',
                         layout=widgets.Layout(width='95%', height='40px'))
sd_output = widgets.Output(layout=widgets.Layout(width='100%', min_height='300px',
                                                  border='1px solid #ddd'))

def on_single_dock(btn):
    btn.disabled = True
    btn.description = 'Running...'
    try:
        run_pro_docking(sd_pdb.value, sd_smiles.value, sd_name.value,
                        sd_scoring.value, sd_engine.value,
                        sd_exhaust.value, sd_poses.value,
                        sd_box.value, sd_residues.value,
                        sd_alpha.value, sd_output)
    finally:
        btn.disabled = False
        btn.description = 'Run Docking'

sd_btn.on_click(on_single_dock)

single_tab = widgets.HBox([
    widgets.VBox([
        widgets.HTML('<h3>Target</h3>'),
        sd_pdb, sd_residues,
        widgets.HTML('<h3>Ligand</h3>'),
        sd_smiles, sd_name,
        widgets.HTML('<h3>Scoring & Engine</h3>'),
        sd_scoring, sd_alpha_box, sd_engine,
        widgets.HTML('<small>GPU acceleration available in Batch mode</small>'),
        widgets.HTML('<h3>Parameters</h3>'),
        sd_exhaust, sd_poses, sd_box,
        sd_btn,
    ], layout=widgets.Layout(width='40%', padding='10px')),
    widgets.VBox([sd_output],
                 layout=widgets.Layout(width='60%', padding='10px'))
])

# === Batch Screening Tab ===
bs_pdb = widgets.Text(value='1HSG', description='PDB ID:', style=style, layout=layout_input)
bs_residues = widgets.Text(value='23,24,25,26,27,28,29,30',
                            description='Active Site:', style=style, layout=layout_input)
bs_scoring = widgets.RadioButtons(options=['Vina', 'Vina + Gnina CNN', 'Consensus'],
                                   value='Vina', description='Scoring:', style=style)
bs_alpha = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05,
                                description='Consensus Alpha:', style=style, layout=layout_input,
                                readout_format='.2f')
bs_alpha_label = widgets.HTML(
    '<small>Alpha controls Vina vs Gnina weight: 0.0 = Gnina only, 0.5 = equal, 1.0 = Vina only</small>')
bs_alpha_box = widgets.VBox([bs_alpha, bs_alpha_label],
                             layout=widgets.Layout(width='95%'))
bs_alpha_box.layout.display = 'none'

def _toggle_bs_alpha(change):
    bs_alpha_box.layout.display = '' if change['new'] == 'Consensus' else 'none'
bs_scoring.observe(_toggle_bs_alpha, names='value')

bs_engine = widgets.RadioButtons(options=['Vina (CPU)', 'Uni-Dock (GPU)'],
                                  value='Vina (CPU)', description='Engine:', style=style)
bs_exhaust = widgets.IntSlider(value=8, min=8, max=64, step=8,
                                description='Exhaustiveness:', style=style, layout=layout_input)
bs_box = widgets.IntSlider(value=20, min=15, max=40, step=5,
                            description='Box Size (A):', style=style, layout=layout_input)
bs_compounds = widgets.Textarea(value=DEFAULT_COMPOUNDS, description='Compounds:',
                                 style=style, layout=widgets.Layout(width='95%', height='200px'),
                                 placeholder='Name, SMILES (one per line)')
bs_btn = widgets.Button(description='Run Batch', button_style='primary',
                         layout=widgets.Layout(width='95%', height='40px'))
bs_output = widgets.Output(layout=widgets.Layout(width='100%', min_height='300px',
                                                  border='1px solid #ddd'))

def on_batch(btn):
    btn.disabled = True
    btn.description = 'Running...'
    try:
        run_pro_batch(bs_pdb.value, bs_compounds.value, bs_scoring.value,
                      bs_engine.value, bs_exhaust.value, bs_box.value,
                      bs_residues.value, bs_alpha.value, bs_output)
    finally:
        btn.disabled = False
        btn.description = 'Run Batch'

bs_btn.on_click(on_batch)

batch_tab = widgets.HBox([
    widgets.VBox([
        widgets.HTML('<h3>Batch Configuration</h3>'),
        bs_pdb, bs_residues, bs_scoring, bs_alpha_box, bs_engine, bs_exhaust, bs_box,
        bs_compounds, bs_btn,
    ], layout=widgets.Layout(width='40%', padding='10px')),
    widgets.VBox([bs_output],
                 layout=widgets.Layout(width='60%', padding='10px'))
])

# === Multi-Protein Tab ===
mp_smiles = widgets.Textarea(
    value='CC(C)(C)NC(=O)[C@@H]1C[C@@H]2CCCN2C(=O)[C@H](CC2=CC=CC=C2)NC(=O)[C@@H](CC2=CC=C(O)C=C2)N1',
    description='Ligand SMILES:', style=style,
    layout=widgets.Layout(width='95%', height='60px')
)
mp_name = widgets.Text(value='Indinavir', description='Ligand Name:', style=style, layout=layout_input)
mp_pdbs = widgets.Textarea(
    value='1HSG:23,24,25,26,27,28,29,30\n4LDE\n6LU7:41,49,142,144,145,163,166',
    description='PDB IDs:', style=style,
    layout=widgets.Layout(width='95%', height='120px'),
    placeholder='One PDB per line. Optional residues: 1HSG:23,24,25'
)
mp_scoring = widgets.RadioButtons(options=['Vina', 'Vina + Gnina CNN', 'Consensus'],
                                   value='Vina', description='Scoring:', style=style)
mp_alpha = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05,
                                description='Consensus Alpha:', style=style, layout=layout_input,
                                readout_format='.2f')
mp_alpha_label = widgets.HTML(
    '<small>Alpha controls Vina vs Gnina weight: 0.0 = Gnina only, 0.5 = equal, 1.0 = Vina only</small>')
mp_alpha_box = widgets.VBox([mp_alpha, mp_alpha_label],
                             layout=widgets.Layout(width='95%'))
mp_alpha_box.layout.display = 'none'

def _toggle_mp_alpha(change):
    mp_alpha_box.layout.display = '' if change['new'] == 'Consensus' else 'none'
mp_scoring.observe(_toggle_mp_alpha, names='value')

mp_box = widgets.IntSlider(value=20, min=15, max=40, step=5,
                            description='Box Size (A):', style=style, layout=layout_input)
mp_exhaust = widgets.IntSlider(value=32, min=8, max=128, step=8,
                                description='Exhaustiveness:', style=style, layout=layout_input)
mp_engine = widgets.Dropdown(options=['Vina (CPU)', 'Uni-Dock (GPU)'], value='Vina (CPU)',
                             description='Engine:', style=style, layout=layout_input)
mp_btn = widgets.Button(description='Run Multi-Protein Docking', button_style='primary',
                         layout=widgets.Layout(width='95%', height='40px'))
mp_output = widgets.Output(layout=widgets.Layout(width='100%', min_height='300px',
                                                  border='1px solid #ddd'))

def on_multi_protein(btn):
    btn.disabled = True
    btn.description = 'Running...'
    try:
        run_pro_multi_protein(mp_smiles.value, mp_name.value, mp_pdbs.value,
                              mp_box.value, mp_exhaust.value,
                              mp_engine.value, mp_scoring.value,
                              mp_alpha.value, mp_output)
    finally:
        btn.disabled = False
        btn.description = 'Run Multi-Protein Docking'

mp_btn.on_click(on_multi_protein)

multi_tab = widgets.HBox([
    widgets.VBox([
        widgets.HTML('<h3>Ligand</h3>'),
        mp_smiles, mp_name,
        widgets.HTML('<h3>Target Proteins</h3>'),
        mp_pdbs,
        widgets.HTML('<small>Format: one PDB ID per line. Optionally add residues: '
                     '<code>1HSG:23,24,25,26</code></small>'),
        widgets.HTML('<h3>Scoring & Engine</h3>'),
        mp_scoring, mp_alpha_box, mp_engine,
        widgets.HTML('<h3>Parameters</h3>'),
        mp_box, mp_exhaust,
        mp_btn,
    ], layout=widgets.Layout(width='40%', padding='10px')),
    widgets.VBox([mp_output],
                 layout=widgets.Layout(width='60%', padding='10px'))
])

# === About Tab ===
about_tab = widgets.HTML(value="""
<div style="max-width:800px; padding:20px; font-family:sans-serif;">
<h3>AcuDock Pro</h3>

<h4>Pipeline</h4>
<pre>
PDB ID --> PDBFixer --> PDBQT
SMILES --> RDKit 3D --> Meeko --> PDBQT
         |                        |
         v                        v
     Vina / Uni-Dock --> Gnina CNN Rescore --> Consensus
         |                                        |
         +---------> 3D Visualization <-----------+
</pre>

<h4>Scoring Modes</h4>
<table border="1" cellpadding="5" style="border-collapse:collapse;">
<tr><th>Mode</th><th>Description</th><th>Accuracy</th></tr>
<tr><td>Vina</td><td>Classical empirical</td><td>~58% redocking</td></tr>
<tr><td>Vina + Gnina CNN</td><td>CNN-based rescoring</td><td>~73% redocking</td></tr>
<tr><td>Consensus</td><td>Z-score weighted (adjustable alpha)</td><td>Best overall</td></tr>
</table>

<h4>Consensus Alpha</h4>
<p>The <b>Alpha</b> slider controls the weighting between Vina and Gnina CNN scores
in Consensus mode. Scores are z-score normalized, then combined:</p>
<pre>consensus = alpha × z(Vina) + (1 − alpha) × z(Gnina_CNN)</pre>
<ul>
<li><b>alpha = 1.0:</b> Vina only</li>
<li><b>alpha = 0.5:</b> Equal weight (default)</li>
<li><b>alpha = 0.0:</b> Gnina CNN only</li>
</ul>

<h4>Engines</h4>
<ul>
<li><b>Vina (CPU):</b> Reliable, no GPU needed</li>
<li><b>Uni-Dock (GPU):</b> 1000x+ speedup for batch screening (10+ compounds)</li>
</ul>

<p><em>MIT License | AcuDock Project</em></p>
</div>
""")

# === Assemble Tabs ===
tabs = widgets.Tab(children=[single_tab, batch_tab, multi_tab, about_tab])
tabs.set_title(0, 'Single Docking')
tabs.set_title(1, 'Batch Screening')
tabs.set_title(2, 'Multi-Protein')
tabs.set_title(3, 'About')

display(widgets.HTML('<h1>AcuDock Pro</h1>'
                     '<p><b>Interactive molecular docking with CNN rescoring and GPU acceleration.</b></p>'))
display(tabs)
